# Lab 2.2 · Data Profiling ข้อมูลลูกค้า (customers.csv)

**ต่อจาก Lab 2.1 · 45 นาที**

ใน Lab 2.1 เราแก้ปัญหา "รูปแบบ" ของ `sales_raw.csv` ไปแล้ว (วันที่ พ.ศ., ชื่อสาขาสะกดต่าง, ราคามีคำว่า "บาท")
แต่ข้อมูลที่ **รูปแบบถูกทุกช่อง ก็ยังผิดได้** Lab นี้จะฝึกหาความผิดปกติ 3 ระดับ

| ระดับ | คำถาม | ตัวอย่าง |
|---|---|---|
| 1 · คอลัมน์ | แต่ละช่องถูกรูปแบบไหม | ค่าว่าง, รหัสซ้ำ, วันที่ผิดรูปแบบ |
| 2 · ข้ามตาราง | ตรงกับตารางอื่นไหม | สมัครก่อนสาขาเปิด, ลูกค้าในยอดขายที่ไม่มีในทะเบียน |
| 3 · ความหมาย | สมเหตุสมผลทางธุรกิจไหม | เบอร์โทรซ้ำ, เพศไม่สัมพันธ์กับชื่อ, ลูกค้าอายุต่ำกว่า 18 |

| ขั้น | เวลา | ทำอะไร |
|---|---|---|
| 1 | 5 นาที | โหลดและดูข้อมูลครั้งแรก |
| 2 | 10 นาที | Profiling ระดับคอลัมน์ |
| 3 | 10 นาที | Profiling ข้ามตาราง (customers × branches × sales) |
| 4 | 10 นาที | ความผิดปกติเชิงความหมาย |
| 5 | 5 นาที | ตัดสินใจว่าจะจัดการแต่ละเรื่องอย่างไร |
| 6 | 5 นาที | Export รายงาน profiling และไฟล์ที่ติด flag |

**หลักสำคัญของ Lab นี้:** รายงานและติด flag เท่านั้น **ไม่ลบ ไม่แก้** ข้อมูลลูกค้า เพราะทะเบียนลูกค้าเป็นข้อมูลต้นทางที่ฝ่ายอื่นใช้ต่อ

**วิธีใช้กับ AI:** ใช้ Gemini ใน Colab หรือคัดลอก prompt ไปใช้กับ Claude / ChatGPT แล้ววางโค้ดกลับมา (ในเซลล์มีโค้ดตัวอย่างให้แล้ว ลองเขียน prompt เองก่อนแล้วเทียบผล)

## ขั้นที่ 0 · เตรียมเครื่องมือ
รันเซลล์นี้ก่อน (ไม่ต้องแก้) มีฟังก์ชัน `check()` สำหรับตรวจคำตอบโดยไม่เฉลยตัวเลข และ `load_csv()` สำหรับอัปโหลดไฟล์ใน Colab

In [ ]:
# ===== เซลล์ตัวช่วยของ Lab 2.2 · รันก่อน ไม่ต้องแก้ =====
import base64, json, os, re
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

_EXP = json.loads(base64.b64decode(
    "eyJyYXdfcm93cyI6IDMwMDAsICJkdXBfaWQiOiAwLCAiYmxhbmtfY2VsbHMiOiAwLCAiYmFkX2RhdGUiOiAwLCAiYmFkX3Bob25lIjogMCwgInVua25vd25fYnJhbmNoIjogMCwgImR1cF9waG9uZV9yb3dzIjogOTksICJqb2luZWRfYmVmb3JlX29wZW4iOiAwLCAib3JwaGFuX3NhbGVzX2lkcyI6IDAsICJuZXZlcl9ib3VnaHQiOiA0OTMsICJtaW5vcnMiOiAxMTN9"
).decode())
_LABEL = {
    "raw_rows": "จำนวนลูกค้าทั้งหมด",
    "dup_id": "customer_id ที่ซ้ำ (ไม่นับตัวแรก)",
    "blank_cells": "จำนวนช่องว่างทั้งไฟล์",
    "bad_date": "joined_date ที่ไม่ใช่ YYYY-MM-DD หรือเป็นวันที่ที่ไม่มีจริง",
    "bad_phone": "phone ที่ไม่ใช่รูปแบบ 0XX-xxx-XXXX",
    "unknown_branch": "home_branch_id ที่ไม่มีใน branches.csv",
    "dup_phone_rows": "แถวที่ใช้เบอร์โทรซ้ำกับคนอื่น (นับทุกแถวในกลุ่ม)",
    "joined_before_open": "ลูกค้าที่สมัครก่อนสาขาของตัวเองเปิด",
    "orphan_sales_ids": "customer_id ใน sales ที่ไม่มีใน customers (นับรหัสไม่ซ้ำ)",
    "never_bought": "ลูกค้าที่ไม่เคยซื้อเลยใน sales",
    "minors": "ลูกค้าอายุต่ำกว่า 18",
}
_HINT = {
    "dup_id": "ใช้ df['customer_id'].duplicated().sum()",
    "blank_cells": "อ่านไฟล์ด้วย keep_default_na=False ค่าว่างจึงเป็น '' ลอง (df.apply(lambda s: s.str.strip()) == '').sum().sum()",
    "bad_date": "pd.to_datetime(..., format='%Y-%m-%d', errors='coerce').isna() จับทั้งรูปแบบผิดและวันที่ที่ไม่มีจริง",
    "bad_phone": "regex r'^0\\d{2}-xxx-\\d{4}$'",
    "dup_phone_rows": "ใช้ duplicated(keep=False) เพื่อนับทุกแถวในกลุ่ม ไม่ใช่แค่ตัวที่สอง",
    "joined_before_open": "merge กับ branches.csv ด้วย home_branch_id = branch_id แล้วเทียบ joined_date < opened_date",
    "orphan_sales_ids": "ตัดแถว customer_id ว่างใน sales ออกก่อน แล้วนับ set ที่ไม่อยู่ใน customers",
    "never_bought": "ลูกค้าใน customers ที่ไม่อยู่ใน sales['customer_id'].unique()",
    "minors": "age_group == 'ต่ำกว่า 18'",
}


def check(name, value):
    """เทียบคำตอบกับเฉลยโดยไม่บอกตัวเลข"""
    exp = _EXP[name]
    label = _LABEL[name]
    if value is None:
        print(f"⬜ {label}: ยังไม่ได้ใส่ค่า")
    elif int(value) == exp:
        print(f"✅ {label}: {int(value):,}")
    elif abs(int(value) - exp) <= max(2, exp * 0.03):
        print(f"🟡 {label}: {int(value):,} ใกล้แล้ว แต่ยังไม่ตรง · {_HINT.get(name, 'ตรวจเงื่อนไขอีกครั้ง')}")
    else:
        print(f"❌ {label}: {int(value):,} ยังไม่ถูก · {_HINT.get(name, 'ตรวจเงื่อนไขอีกครั้ง')}")


def load_csv(name):
    """อ่าน CSV ทุกคอลัมน์เป็นข้อความ ถ้าไม่มีไฟล์จะให้อัปโหลด (Google Colab)"""
    if not os.path.exists(name):
        try:
            from google.colab import files
            print(f"ไม่พบ {name} กรุณาอัปโหลดไฟล์")
            files.upload()
        except ImportError:
            raise FileNotFoundError(f"ไม่พบไฟล์ {name} ให้วางไฟล์ไว้โฟลเดอร์เดียวกับ notebook")
    return pd.read_csv(name, dtype=str, keep_default_na=False, encoding="utf-8-sig")


AGE_ORDER = ["ต่ำกว่า 18", "18-24", "25-34", "35-44", "45-54", "55+"]
STD_GENDER = ["ชาย", "หญิง", "ไม่ระบุ"]
print("พร้อมแล้ว")

## ขั้นที่ 1 · ดูข้อมูลครั้งแรก (5 นาที)

อัปโหลด 3 ไฟล์ (ใน Colab จะมีปุ่มให้อัปโหลดทีละไฟล์)
- `customers.csv` · ทะเบียนสมาชิก ไฟล์หลักของ Lab นี้
- `branches.csv` · ข้อมูลสาขาและวันเปิด
- `sales.csv` · ยอดขายที่ทำความสะอาดแล้วจาก Lab 2.1 (`sales_clean.csv` ที่เปลี่ยนชื่อแล้ว หรือไฟล์ใน `public/` ของ Dashboard)

อ่าน **ทุกคอลัมน์เป็นข้อความ** เหมือน Lab 2.1 เพื่อไม่ให้ pandas ซ่อนปัญหา

In [ ]:
cust = load_csv("customers.csv")
branches = load_csv("branches.csv")
sales = load_csv("sales.csv")

print(f"customers: {len(cust):,} แถว × {cust.shape[1]} คอลัมน์")
print(f"branches : {len(branches):,} แถว · sales: {len(sales):,} แถว")
cust.head(8)

**คำถามชวนคิด:** คอลัมน์ไหนในไฟล์นี้เป็น **ข้อมูลส่วนบุคคล** ตาม PDPA บ้าง และทำไมเบอร์โทรถึงถูกปิดเลขกลางเป็น `xxx`

## ขั้นที่ 2 · Profiling ระดับคอลัมน์ (10 นาที)

**Prompt**
```
ช่วยทำ data profiling ของ DataFrame cust ด้วย pandas (ทุกคอลัมน์เป็น str ค่าว่างเป็น "")
คอลัมน์: customer_id, nickname, gender, age_group, home_branch_id, joined_date, phone
- จำนวนค่าว่าง ค่าไม่ซ้ำ และค่าที่มีช่องว่างหน้า/ท้าย ของแต่ละคอลัมน์
- customer_id ซ้ำไหม รูปแบบ C + เลข 5 หลักครบทุกแถวไหม และเลขรันต่อเนื่องหรือมีเลขหาย
- ค่าทั้งหมดของ gender, age_group, home_branch_id พร้อมจำนวน
- joined_date ที่ไม่ใช่ YYYY-MM-DD หรือเป็นวันที่ไม่มีจริง พร้อมช่วงวันที่ต่ำสุด–สูงสุด
- phone ที่ไม่ใช่รูปแบบ 0XX-xxx-XXXX
เก็บจำนวนไว้ในตัวแปร n_dup_id, n_blank, n_bad_date, n_bad_phone
อย่าแก้ข้อมูล ให้รายงานอย่างเดียว
```

In [ ]:
# วางโค้ดที่ AI เขียนให้ที่นี่ แล้วรัน (ด้านล่างคือโค้ดตัวอย่าง)
print(f"ขนาดข้อมูล: {len(cust):,} แถว × {cust.shape[1]} คอลัมน์\n")

# 1) ค่าว่าง / ค่าไม่ซ้ำ / ช่องว่างหน้า-ท้าย
summary = pd.DataFrame({
    "ค่าว่าง": (cust.apply(lambda s: s.str.strip()) == "").sum(),
    "ค่าไม่ซ้ำ": cust.nunique(),
    "มีช่องว่างหน้า/ท้าย": (cust != cust.apply(lambda s: s.str.strip())).sum(),
})
print("== 1) ภาพรวมคอลัมน์ ==\n", summary, "\n")
n_blank = int(summary["ค่าว่าง"].sum())

# 2) customer_id
n_dup_id = int(cust["customer_id"].duplicated().sum())
bad_id = ~cust["customer_id"].str.match(r"^C\d{5}$")
nums = cust.loc[~bad_id, "customer_id"].str[1:].astype(int)
missing_ids = sorted(set(range(1, nums.max() + 1)) - set(nums))
print(f"== 2) customer_id ซ้ำ {n_dup_id:,} · รูปแบบผิด {bad_id.sum():,} · ช่วง C{nums.min():05d}–C{nums.max():05d} · เลขที่หาย {len(missing_ids):,}\n")

# 3) ค่าหมวดหมู่
for col in ["gender", "age_group", "home_branch_id"]:
    print(f"== 3) {col} ==")
    print(cust[col].value_counts().to_string(), "\n")
print("ค่า gender นอกมาตรฐาน:", (~cust["gender"].isin(STD_GENDER)).sum(),
      "· ค่า age_group นอกมาตรฐาน:", (~cust["age_group"].isin(AGE_ORDER)).sum(), "\n")

# 4) joined_date
jd = pd.to_datetime(cust["joined_date"], format="%Y-%m-%d", errors="coerce")
n_bad_date = int(jd.isna().sum())
print(f"== 4) joined_date ผิด {n_bad_date:,} · ตัวอย่าง {cust.loc[jd.isna(), 'joined_date'].head(5).tolist()}")
print(f"ช่วงวันที่: {jd.min():%Y-%m-%d} ถึง {jd.max():%Y-%m-%d}\n")

# 5) phone
n_bad_phone = int((~cust["phone"].str.match(r"^0\d{2}-xxx-\d{4}$")).sum())
print(f"== 5) phone ผิดรูปแบบ {n_bad_phone:,}")
print("prefix:", cust["phone"].str[:3].value_counts().sort_index().to_dict(), "\n")

print(dict(n_dup_id=n_dup_id, n_blank=n_blank, n_bad_date=n_bad_date, n_bad_phone=n_bad_phone))

### ✔️ ตรวจคำตอบระดับคอลัมน์

**คำถามชวนคิด:** ถ้าได้ ✅ ทุกข้อและทุกค่าเป็น 0 แปลว่าข้อมูล "สะอาด" แล้วหรือยัง

In [ ]:
check("raw_rows", len(cust))
check("dup_id", n_dup_id)
check("blank_cells", n_blank)
check("bad_date", n_bad_date)
check("bad_phone", n_bad_phone)

## ขั้นที่ 3 · Profiling ข้ามตาราง (10 นาที)

ข้อมูลที่รูปแบบถูกอาจ **ขัดกับตารางอื่น** ได้ ตรวจความสอดคล้องระหว่าง `customers`, `branches` และ `sales`

**Prompt**
```
มี DataFrame 3 ตัว (ทุกคอลัมน์เป็น str ค่าว่างเป็น "")
- cust: customer_id, nickname, gender, age_group, home_branch_id, joined_date (YYYY-MM-DD), phone
- branches: branch_id, branch (ชื่อสาขา), branch_type, lat, lng, opened_date
- sales: order_id, datetime (YYYY-MM-DDTHH:MM:SS+07:00), branch (ชื่อสาขา), product_id, qty, unit_price, customer_id (ว่าง = ลูกค้าทั่วไป), ...
ช่วยตรวจ (รายงานอย่างเดียว ห้ามแก้ข้อมูล)
1. home_branch_id ที่ไม่มีใน branches
2. ลูกค้าที่ joined_date ก่อน opened_date ของสาขาตัวเอง
3. customer_id ใน sales ที่ไม่มีใน cust (นับรหัสไม่ซ้ำ)
4. ลูกค้าที่ไม่เคยซื้อเลย
5. ลูกค้าที่ซื้อครั้งแรกก่อนวันสมัคร (ใช้ 10 ตัวอักษรแรกของ datetime เป็นวันที่ ห้ามใช้ pd.to_datetime เพราะจะแปลงเป็น UTC)
6. สัดส่วนลูกค้าที่ home_branch ตรงกับสาขาที่ซื้อบ่อยที่สุด
7. วันสมัครเร็วสุดของแต่ละสาขา เทียบกับวันเปิดสาขา และวันแรกของ sales
เก็บจำนวนไว้ในตัวแปร n_unknown_branch, n_joined_before_open, n_orphan_ids, n_never_bought, n_buy_before_join
```

In [ ]:
# วางโค้ดที่ AI เขียนให้ที่นี่ แล้วรัน (ด้านล่างคือโค้ดตัวอย่าง)
br = branches.set_index("branch_id")
name_to_id = dict(zip(branches["branch"], branches["branch_id"]))

# 1) สาขาที่ไม่มีจริง
n_unknown_branch = int((~cust["home_branch_id"].isin(br.index)).sum())
print(f"1) home_branch_id ที่ไม่มีใน branches: {n_unknown_branch:,}")

# 2) สมัครก่อนสาขาเปิด (สตริง YYYY-MM-DD เทียบกันได้ตรง ๆ)
opened = cust["home_branch_id"].map(br["opened_date"])
before_open = cust["joined_date"] < opened
n_joined_before_open = int(before_open.sum())
print(f"2) สมัครก่อนสาขาเปิด: {n_joined_before_open:,}")

# 3–5) เทียบกับ sales
s = sales[sales["customer_id"].str.strip() != ""].copy()
s["date"] = s["datetime"].str[:10]            # ไม่ใช้ pd.to_datetime กัน timezone เลื่อน
ids_cust, ids_sales = set(cust["customer_id"]), set(s["customer_id"])
n_orphan_ids = len(ids_sales - ids_cust)
never = cust[~cust["customer_id"].isin(ids_sales)]
n_never_bought = len(never)
print(f"3) customer_id ใน sales ที่ไม่มีในทะเบียน: {n_orphan_ids:,}")
print(f"4) ลูกค้าที่ไม่เคยซื้อ: {n_never_bought:,} ({n_never_bought / len(cust):.1%})")

first_buy = s.groupby("customer_id")["date"].min()
j = cust.set_index("customer_id").join(first_buy.rename("first_buy"), how="inner")
n_buy_before_join = int((j["first_buy"] < j["joined_date"]).sum())
print(f"5) ซื้อครั้งแรกก่อนวันสมัคร: {n_buy_before_join:,}")
print(f"   ระยะจากวันสมัครถึงซื้อครั้งแรก (วัน): ค่ากลาง "
      f"{(pd.to_datetime(j['first_buy']) - pd.to_datetime(j['joined_date'])).dt.days.median():.0f}")

# 6) home branch ตรงกับสาขาที่ซื้อบ่อยสุดไหม
s["branch_id"] = s["branch"].map(name_to_id)
# นับเป็น "บิล" ต่อสาขา (1 บิลมีหลายแถว) และถ้าสาขาตัวเองเสมอกับสาขาอื่น ถือว่ายังตรง
bills = s.groupby(["customer_id", "branch_id"])["order_id"].nunique().unstack(fill_value=0)
hb = cust.set_index("customer_id")["home_branch_id"].reindex(bills.index)
home_bills = pd.Series([bills.at[c, h] if h in bills.columns else 0 for c, h in hb.items()], index=bills.index)
match = home_bills >= bills.max(axis=1)      # True = ซื้อสาขาตัวเองบ่อยที่สุด (หรือเท่ากับสาขาอื่น)
print(f"6) ซื้อสาขาตัวเองบ่อยที่สุด (หรือเท่ากัน): {match.mean():.1%} "
      f"(ซื้อสาขาอื่นบ่อยกว่า {(~match).sum():,} คน · ในนี้ไม่เคยซื้อสาขาตัวเองเลย {((~match) & (home_bills == 0)).sum():,} คน)")

# 7) วันสมัครเร็วสุดของแต่ละสาขา
first_join = cust.groupby("home_branch_id")["joined_date"].agg(["min", "max", "count"])
first_join["opened_date"] = br["opened_date"]
print(f"\n7) วันแรกของ sales: {sales['datetime'].str[:10].min()} · วันสุดท้าย: {sales['datetime'].str[:10].max()}")
first_join

### ✔️ ตรวจคำตอบข้ามตาราง

In [ ]:
check("unknown_branch", n_unknown_branch)
check("joined_before_open", n_joined_before_open)
check("orphan_sales_ids", n_orphan_ids)
check("never_bought", n_never_bought)

**คำถามชวนคิด (ดูตารางข้อ 7):** สาขาสยามเปิดตั้งแต่ปี 2023 แต่ทำไม **ไม่มีสมาชิกคนไหนสมัครก่อน 2025-04-01** เลย
เป็นไปได้กี่สาเหตุ และควรถามใครก่อนจะสรุปว่า "สมาชิกใหม่ปีแรกโตเร็ว"

## ขั้นที่ 4 · ความผิดปกติเชิงความหมาย (10 นาที)

ส่วนนี้โค้ดหาตัวเลขได้ แต่ **คนต้องตีความ** ว่าผิดปกติจริงหรือเป็นเรื่องที่คาดได้

### 4.1 เบอร์โทรซ้ำ = ลูกค้าซ้ำหรือเปล่า

**Prompt**
```
ใน cust ให้หาแถวที่ phone ซ้ำกับคนอื่น (นับทุกแถวในกลุ่ม) แสดงตัวอย่างกลุ่มที่ซ้ำ
และบอกว่าในแต่ละกลุ่ม nickname / gender / age_group / home_branch_id เหมือนกันกี่กลุ่ม
เบอร์ถูกปิดเลขกลางเป็น xxx เหลือ prefix 3 หลัก + เลขท้าย 4 หลัก
ช่วยคำนวณด้วยว่าถ้าเบอร์สุ่มจริง ๆ จะคาดว่ามีแถวที่ชนกันกี่แถว (birthday problem)
เก็บจำนวนแถวที่ซ้ำไว้ใน n_dup_phone_rows
```

In [ ]:
dup_mask = cust["phone"].duplicated(keep=False)
n_dup_phone_rows = int(dup_mask.sum())
groups = cust[dup_mask].groupby("phone")
same = groups.agg(
    คน=("customer_id", "size"),
    ชื่อเหมือน=("nickname", lambda x: x.nunique() == 1),
    เพศเหมือน=("gender", lambda x: x.nunique() == 1),
    อายุเหมือน=("age_group", lambda x: x.nunique() == 1),
    สาขาเหมือน=("home_branch_id", lambda x: x.nunique() == 1),
)
all_same = same[["ชื่อเหมือน", "เพศเหมือน", "อายุเหมือน", "สาขาเหมือน"]].all(axis=1)
print(f"แถวที่เบอร์ซ้ำ: {n_dup_phone_rows:,} ใน {len(same):,} กลุ่ม · กลุ่มที่ข้อมูลเหมือนกันทุกอย่าง: {all_same.sum():,}")

# ถ้าเบอร์สุ่ม: จำนวนแบบที่เป็นไปได้ = จำนวน prefix × 10,000
N = cust["phone"].str[:3].nunique() * 10_000
n = len(cust)
expected = n * (1 - (1 - 1 / N) ** (n - 1))
print(f"จำนวนเบอร์ที่เป็นไปได้หลังปิดเลข: {N:,} · ถ้าสุ่มจะคาดว่าชนกัน ≈ {expected:.0f} แถว")

display(cust[dup_mask].sort_values("phone").head(10))

In [ ]:
check("dup_phone_rows", n_dup_phone_rows)

**คำถามชวนคิด:** จำนวนที่ซ้ำจริงใกล้กับค่าที่คาดจากการสุ่มแค่ไหน ถ้าใกล้กัน แปลว่าอะไร
แล้วถ้าทีมการตลาดอยาก "รวมบัญชีลูกค้าที่เบอร์เดียวกัน" ควรตอบเขาว่าอะไร

### 4.2 เพศสัมพันธ์กับชื่อเล่นไหม

ชื่ออย่าง "สมชาย" หรือ "อัญชลี" บอกเพศได้ค่อนข้างชัด ถ้าข้อมูลถูกต้อง ชื่อเหล่านี้ควรเป็นเพศเดียวเกือบทั้งหมด

In [ ]:
ct = pd.crosstab(cust["nickname"], cust["gender"])[STD_GENDER]
ct["%หญิง"] = (ct["หญิง"] / ct[["ชาย", "หญิง"]].sum(axis=1) * 100).round(1)
ct = ct.sort_values("%หญิง")
print(f"%หญิง ต่อชื่อ: ต่ำสุด {ct['%หญิง'].min()}% · สูงสุด {ct['%หญิง'].max()}% · ภาพรวม "
      f"{(cust['gender'] == 'หญิง').sum() / cust['gender'].isin(['ชาย', 'หญิง']).sum():.1%}")
ct.loc[[x for x in ["สมชาย", "มานพ", "วีระ", "กิตติ", "อัญชลี", "สุดา", "รัตนา", "พิม"] if x in ct.index]]

**คำถามชวนคิด:** ถ้าทุกชื่อมี %หญิง ใกล้ค่าภาพรวม แปลว่าคอลัมน์ไหน (gender หรือ nickname) เชื่อถือไม่ได้
ควรใช้ gender วิเคราะห์ "เมนูที่ผู้หญิงชอบ" ต่อหรือไม่

### 4.3 อายุ และการสมัครรายเดือน

In [ ]:
n_minors = int((cust["age_group"] == "ต่ำกว่า 18").sum())
print(f"ลูกค้าอายุต่ำกว่า 18: {n_minors:,} คน ({n_minors / len(cust):.1%})")

age_branch = pd.crosstab(cust["age_group"], cust["home_branch_id"], normalize="columns").reindex(AGE_ORDER)
age_branch.columns = [f"{c} {br.loc[c, 'branch_type']}" for c in age_branch.columns]
display((age_branch * 100).round(1))

monthly = cust["joined_date"].str[:7].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(monthly.index, monthly.values, color=["#9DC3C4" if m == monthly.index[-1] else "#0F8B8D" for m in monthly.index])
ax.set_ylabel("New members"); ax.set_title("New members per month (last month = partial)")
ax.tick_params(axis="x", rotation=60, labelsize=8)
plt.tight_layout(); plt.show()
# หมายเหตุ: ใช้ป้ายภาษาอังกฤษ เพราะ matplotlib ใน Colab ไม่มีฟอนต์ไทย
print(f"เดือนสุดท้าย {monthly.index[-1]} มีสมาชิกใหม่ {monthly.iloc[-1]:,} คน ข้อมูลถึงวันที่ {cust['joined_date'].max()}")

In [ ]:
check("minors", n_minors)

**คำถามชวนคิด:**
- สาขามหาวิทยาลัย (สถานศึกษา) ควรมีลูกค้าอายุ 18-24 มากกว่าสาขาอื่นชัดเจน ข้อมูลเป็นอย่างนั้นไหม
- ลูกค้าอายุต่ำกว่า 18 ที่มีเบอร์โทรในระบบ ต้องระวังอะไรตาม PDPA ก่อนส่ง SMS โปรโมชัน

## ขั้นที่ 5 · ตัดสินใจก่อนลงมือ (5 นาที · คุยกับเพื่อนข้าง ๆ)

ดับเบิลคลิกเซลล์นี้เพื่อกรอกตาราง

| สิ่งที่พบ | ทางเลือก | ตัดสินใจ | เหตุผล |
|---|---|---|---|
| เบอร์โทรซ้ำ | รวมบัญชี / ติด flag / ไม่ทำอะไร | | |
| เพศไม่สัมพันธ์กับชื่อ | ใช้ต่อ / ใช้แบบมีข้อจำกัด / ไม่ใช้ | | |
| สมาชิกเริ่ม 2025-04-01 ทุกสาขา | ถามเจ้าของข้อมูล / สรุปว่าโปรแกรมเพิ่งเริ่ม | | |
| สมาชิกที่ไม่เคยซื้อ | ลบ / เก็บ + ติด flag สำหรับแคมเปญ | | |
| home_branch ≠ สาขาที่ซื้อจริง | ใช้ home_branch / ใช้สาขาที่ซื้อจริง | | |
| ลูกค้าอายุต่ำกว่า 18 | ติด flag ห้ามส่งการตลาด / ไม่ทำอะไร | | |

> **สำหรับผู้สอน:** เติมข้อมูลจากร้านตรงนี้ก่อนแจกนักศึกษา เช่น เบอร์โทรถูกปิดเลขกลางตั้งแต่ตอน export หรือไม่ ·
> โปรแกรมสมาชิกเริ่มเมื่อไร หรือมีการย้ายระบบที่ทำให้ joined_date ของสมาชิกเก่าเปลี่ยนไปหรือไม่

## ขั้นที่ 6 · Export รายงาน profiling และไฟล์ที่ติด flag

ไม่ลบหรือแก้แถวไหนเลย แค่เพิ่มคอลัมน์ `flag_*` ให้ทีมที่ใช้ต่อเลือกกรองเองได้

**Prompt**
```
จากผลด้านบน ช่วยสร้าง
1. profiling_log: DataFrame (ระดับ, สิ่งที่ตรวจ, จำนวน, ระดับความรุนแรง, ข้อเสนอแนะ)
2. flagged = cust.copy() แล้วเพิ่มคอลัมน์ True/False: flag_dup_phone, flag_minor, flag_never_bought, flag_home_ne_top_branch
   ห้ามลบหรือแก้ค่าในคอลัมน์เดิม จำนวนแถวต้องเท่าเดิม
บันทึกเป็น customer_profiling_log.csv และ customers_flagged.csv (utf-8-sig) แล้วดาวน์โหลดใน Colab
```

In [ ]:
def row(level, item, n, severity, advice):
    return {"ระดับ": level, "สิ่งที่ตรวจ": item, "จำนวน": int(n), "ความรุนแรง": severity, "ข้อเสนอแนะ": advice}

profiling_log = pd.DataFrame([
    row("คอลัมน์", "customer_id ซ้ำ", n_dup_id, "ผ่าน", "-"),
    row("คอลัมน์", "ช่องว่าง", n_blank, "ผ่าน", "-"),
    row("คอลัมน์", "joined_date ผิดรูปแบบ", n_bad_date, "ผ่าน", "-"),
    row("คอลัมน์", "phone ผิดรูปแบบ", n_bad_phone, "ผ่าน", "-"),
    row("ข้ามตาราง", "home_branch_id ไม่มีใน branches", n_unknown_branch, "ผ่าน", "-"),
    row("ข้ามตาราง", "สมัครก่อนสาขาเปิด", n_joined_before_open, "ผ่าน", "-"),
    row("ข้ามตาราง", "customer_id ใน sales ไม่มีในทะเบียน", n_orphan_ids, "ผ่าน", "-"),
    row("ข้ามตาราง", "ซื้อครั้งแรกก่อนวันสมัคร", n_buy_before_join, "ผ่าน", "-"),
    row("ข้ามตาราง", "สมาชิกที่ไม่เคยซื้อ", n_never_bought, "ข้อสังเกต",
        "เก็บไว้ ใช้เป็นกลุ่มเป้าหมายแคมเปญกระตุ้นการซื้อครั้งแรก"),
    row("ข้ามตาราง", "ซื้อสาขาอื่นบ่อยกว่า home_branch (บิล)", (~match).sum(), "ข้อสังเกต",
        "วิเคราะห์ระดับสาขาให้ระบุว่าใช้สาขาที่สมัครหรือสาขาที่ซื้อจริง"),
    row("ข้ามตาราง", "สมาชิกทุกคนสมัครตั้งแต่ 2025-04-01", (cust["joined_date"] >= "2025-04-01").sum(), "เตือน",
        "ยืนยันกับเจ้าของข้อมูลก่อนใช้ joined_date วัดการเติบโตช่วงแรก"),
    row("ความหมาย", "เบอร์โทรซ้ำ (แถว)", n_dup_phone_rows, "ข้อสังเกต",
        f"ใกล้ค่าที่คาดจากการสุ่ม (~{expected:.0f}) เพราะปิดเลขกลาง ห้ามใช้เบอร์รวมบัญชี"),
    row("ความหมาย", "เพศไม่สัมพันธ์กับชื่อเล่น (จำนวนชื่อ)", len(ct), "เตือน",
        "gender หรือ nickname เชื่อถือไม่ได้ อย่าใช้วิเคราะห์ตามเพศโดยไม่ระบุข้อจำกัด"),
    row("ความหมาย", "ลูกค้าอายุต่ำกว่า 18", n_minors, "เตือน",
        "ติด flag ห้ามส่งการตลาดจนกว่าจะมีความยินยอมจากผู้ปกครอง (PDPA)"),
])

flagged = cust.copy()
flagged["flag_dup_phone"] = dup_mask
flagged["flag_minor"] = flagged["age_group"] == "ต่ำกว่า 18"
flagged["flag_never_bought"] = ~flagged["customer_id"].isin(ids_sales)
flagged["flag_home_ne_top_branch"] = flagged["customer_id"].map(~match).fillna(False).astype(bool)

assert len(flagged) == len(cust) and flagged[cust.columns].equals(cust), "ห้ามแก้ข้อมูลเดิม"
display(profiling_log)
print(flagged.filter(like="flag_").sum().to_string())

In [ ]:
profiling_log.to_csv("customer_profiling_log.csv", index=False, encoding="utf-8-sig")
flagged.to_csv("customers_flagged.csv", index=False, encoding="utf-8-sig")
print("บันทึก customer_profiling_log.csv และ customers_flagged.csv แล้ว")

try:
    from google.colab import files
    files.download("customer_profiling_log.csv")
    files.download("customers_flagged.csv")
except ImportError:
    pass

---
### 🚀 เสร็จเร็ว? ลองต่อ
- ให้ AI เขียนฟังก์ชัน `profile_customers(cust, branches, sales)` ที่คืน `profiling_log` เรียกครั้งเดียวจบ (ใช้ต่อในคาบ pipeline อัตโนมัติ)
- ทำ RFM อย่างง่าย (Recency, Frequency, Monetary) ของสมาชิกที่เคยซื้อ แล้วดูว่ากลุ่ม `flag_home_ne_top_branch` ใช้จ่ายต่างจากกลุ่มอื่นไหม
- ใช้ `customers_flagged.csv` ใน Dashboard: เพิ่ม KPI "สมาชิกที่ยังไม่เคยซื้อ" พร้อมหมายเหตุเรื่องช่วงวันที่ของ joined_date